# DS 227 &middot; Knowledge Discovery in Data &mdash; Week 9 Lab
## Finding a Story in Data

EDA is not a checklist &mdash; it is a hunt for a **claim you can defend**. This week you
turn a question into a comparison, and a comparison into a one-sentence finding.

**How long:** about 45 minutes. Nothing to install.

Work top to bottom. The Stretch section at the end is optional.

---
## Part 0 &middot; Start with a question

A story starts with a question, not a dataset. Here: *does study time relate to passing?*
Run the cell.

In [ ]:
import pandas as pd
df = pd.DataFrame({
    "student": list("ABCDEFGHIJ"),
    "study_hours": [1, 6, 2, 8, 3, 7, 2, 9, 4, 5],
    "section": ["A","A","B","B","A","B","A","B","A","B"],
    "score": [58, 84, 61, 96, 66, 90, 60, 98, 72, 80],
})
df["passed"] = df["score"] >= 75
print(df)

**Answer here** (double-click to edit):

1. State the question this data could answer in one sentence. What is the outcome you care
   about, and what might explain it?
   &rarr; *your answer*

2. We turned `score` into a boolean `passed`. What did that make easier to ask, and what
   nuance did it throw away?
   &rarr; *your answer*

---
## Part 1 &middot; Compare the groups

A finding usually lives in a **comparison** between segments. Run the cell.

In [ ]:
import pandas as pd
df = pd.DataFrame({
    "study_hours": [1, 6, 2, 8, 3, 7, 2, 9, 4, 5],
    "passed":      [False,True,False,True,False,True,False,True,False,True],
})
print(df.groupby("passed")["study_hours"].agg(["mean", "min", "max", "count"]))

**Answer here:**

1. Compare the mean `study_hours` of those who passed and those who did not. How big is the
   gap, and in which direction?
   &rarr; *your answer*

2. The groups are small (`count`). Why does a small sample mean you should hedge your claim
   rather than state it as certain?
   &rarr; *your answer*

---
## Part 2 &middot; Show the contrast

A chart makes the comparison land. Run the cell.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.DataFrame({
    "study_hours": [1, 6, 2, 8, 3, 7, 2, 9, 4, 5],
    "passed":      [False,True,False,True,False,True,False,True,False,True],
})
means = df.groupby("passed")["study_hours"].mean()
fig, ax = plt.subplots()
ax.bar(["failed", "passed"], means)
ax.set_ylabel("Avg study hours"); ax.set_title("Study time by outcome")
plt.show()

**Answer here:**

1. The bars start at zero (as they must). What does the height difference say, at a glance,
   that the table of numbers said more slowly?
   &rarr; *your answer*

2. Could something *other* than study time explain the gap (a confounder)? Name one, and
   how you might check it.
   &rarr; *your answer*

---
## Part 3 &middot; Write the finding

A story is one defensible sentence with a number and a caveat. Run the cell, then write
yours in the answer.

In [ ]:
import pandas as pd
df = pd.DataFrame({
    "study_hours": [1, 6, 2, 8, 3, 7, 2, 9, 4, 5],
    "passed":      [False,True,False,True,False,True,False,True,False,True],
})
g = df.groupby("passed")["study_hours"].mean()
print(f"passed avg {g[True]:.1f}h vs failed avg {g[False]:.1f}h "
      f"(gap {g[True]-g[False]:.1f}h, n={len(df)})")

**Answer here:**

1. Write your finding as **one sentence**: the claim, the number, and the caveat. Example
   shape: *"Students who passed studied about X hours more on average, though with only N
   students this is suggestive, not conclusive."*
   &rarr; *your answer*

2. What single next step would most strengthen this finding &mdash; more data, a controlled
   comparison, or checking a confounder? Why that one?
   &rarr; *your answer*

---
## Stretch &mdash; optional

Stop here if you like; the required part is done.

### Stretch 1 &middot; Check a confounder

Bring back the `section` column from Part 0. Does the pass rate differ by section?
Group by `section` and compare &mdash; could section be driving the result?

In [ ]:
# your code here

### Stretch 2 &middot; Quantify the gap differently

Instead of comparing means, compare the *pass rate* of high-study (>= 5h) vs low-study
students. Does the story survive this second framing?

In [ ]:
# your code here

---
## Submitting

Run the cell below. It uploads this notebook straight from Colab &mdash; nothing to
download.

You need a **submit token** &mdash; one covers every lab for a month. Open
[https://portal.latarak.com/student/submit-token](https://portal.latarak.com/student/submit-token), sign in and generate it,
then add it **once** to Colab's Secrets panel (the &#128273; icon, left sidebar) as
`LATARAK_TOKEN`. After that the cell reads it automatically, with no prompt. No Secrets
panel? The cell will just ask, hiding what you type.

In [ ]:
# --- Submit this notebook ------------------------------------------------------
# Colab only. Anywhere else, use the manual route described below this cell.
import getpass, json, urllib.request, urllib.error

PORTAL, COURSE, WEEK = "https://portal.latarak.com", "ds227", 9

try:
    from google.colab import _message
except ImportError:
    raise SystemExit(
        "Not running in Colab. Download this notebook "
        "(File > Download > Download .ipynb) and upload it at "
        "https://portal.latarak.com/course/ds227/lab/9/submit"
    )

# The LIVE notebook, including edits you have not saved yet.
nb = _message.blocking_request("get_ipynb", timeout_sec=90)["ipynb"]

# A month-long token. Store it once in Colab Secrets (key LATARAK_TOKEN) and
# this reads it with no prompt; otherwise it asks and hides what you type.
try:
    from google.colab import userdata
    token = (userdata.get("LATARAK_TOKEN") or "").strip()
except Exception:
    token = ""
if not token:
    token = getpass.getpass("Submit token (hidden as you type): ").strip()

req = urllib.request.Request(
    PORTAL + "/api/labs/" + COURSE + "/submit-notebook",
    data=json.dumps({"week": WEEK, "notebook": nb}).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer " + token},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.load(r)
    print("Submitted", out["course"], "week", out["week"], "for", out["student"])
    print(out["cells"], "cells,", out["executed"], "executed")
    print(out["message"])
except urllib.error.HTTPError as e:
    print("Not submitted:", json.loads(e.read()).get("error", e.reason))

Prefer to do it by hand? **File &rarr; Download &rarr; Download .ipynb**, then go to the
[Week 9 submission page](https://portal.latarak.com/course/ds227/lab/9/submit) and upload it.

Re-submitting replaces your previous attempt; the most recent version is the one kept.